¡Hola, Luis!

Mi nombre es Tonatiuh Cruz. Me complace revisar tu proyecto hoy.

Al identificar cualquier error inicialmente, simplemente los destacaré. Te animo a localizar y abordar los problemas de forma independiente como parte de tu preparación para un rol como data-scientist. En un entorno profesional, tu líder de equipo seguiría un enfoque similar. Si encuentras la tarea desafiante, proporcionaré una pista más específica en la próxima iteración.

Encontrarás mis comentarios a continuación - **por favor no los muevas, modifiques o elimines**.

Puedes encontrar mis comentarios en cajas verdes, amarillas o rojas como esta:

<div class="alert alert-block alert-success">
<b>Comentario del revisor</b> <a class="tocSkip"></a>

Éxito. Todo está hecho correctamente.
</div>

<div class="alert alert-block alert-warning">
<b>Comentario del revisor</b> <a class="tocSkip"></a>

Observaciones. Algunas recomendaciones.
</div>

<div class="alert alert-block alert-danger">
<b>Comentario del revisor</b> <a class="tocSkip"></a>

Necesita corrección. El bloque requiere algunas correcciones. El trabajo no puede ser aceptado con comentarios en rojo.
</div>

Puedes responderme utilizando esto:


<div class="alert alert-block alert-info">
<b>Respuesta del estudiante.</b> <a class="tocSkip"></a>
</div>

In [1]:
 #librerias necesarias para trabajar. 
import pandas as pd 
from sklearn.metrics import f1_score, roc_auc_score, recall_score, accuracy_score, precision_score, roc_curve, auc
from sklearn.linear_model import LogisticRegression 
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

In [2]:
#abrimos el archivo 

data=pd.read_csv('/datasets/Churn.csv')
#revisamos forma, tipo de datos, numero de entradas, columnas de datos. 
print(data.shape)
print('---------------------separador---------------------')
print(data.info())

(10000, 14)
---------------------separador---------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           9091 non-null   float64
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(3), int64(8), object(3)
memory usage: 1.1+ MB
None


<div class="alert alert-block alert-success">
<b>Comentario del revisor</b> <a class="tocSkip"></a>

Buen trabajo tanto con la carga de librerías y datos como con la exploración inicial de la información.

In [3]:
#analizamos variables para categoricas
print(data['Geography'].unique())
print(data['Gender'].unique())
print(data['Tenure'].unique())
print(data['Tenure'].isna())
#rellenamos los nan con la mediana 
data.fillna(data['Tenure'].median(), inplace=True)

['France' 'Spain' 'Germany']
['Female' 'Male']
[ 2.  1.  8.  7.  4.  6.  3. 10.  5.  9.  0. nan]
0       False
1       False
2       False
3       False
4       False
        ...  
9995    False
9996    False
9997    False
9998    False
9999     True
Name: Tenure, Length: 10000, dtype: bool


Debido a la cantidad de datos faltantes se rellenan con la media para no crear falsos outliers. 

In [4]:
#hacemos one-hot encoding 
one= pd.get_dummies(data, columns=['Geography','Gender'],drop_first=True)
print(one.columns)
print('---------------------separador---------------------')
#separamos caracteristicas. 
features= one.drop(['RowNumber','CustomerId','Surname','Exited'],axis=1)

target=one['Exited']


print('------Porcentaje de clases------')
print("NaNs en features:", features.isna().sum().sum())
print(target.value_counts(normalize=True))
print('---------------------separador---------------------')
#comprobamos que tengamos el mismo numero de entradas.
one.info()
#separamos los df, en entrenamieto y validacion.
features_train, features_valid, target_train, target_valid = train_test_split(features, target, test_size=.25,random_state=12345)


Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Age', 'Tenure',
       'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember',
       'EstimatedSalary', 'Exited', 'Geography_Germany', 'Geography_Spain',
       'Gender_Male'],
      dtype='object')
---------------------separador---------------------
------Porcentaje de clases------
NaNs en features: 0
0    0.7963
1    0.2037
Name: Exited, dtype: float64
---------------------separador---------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   RowNumber          10000 non-null  int64  
 1   CustomerId         10000 non-null  int64  
 2   Surname            10000 non-null  object 
 3   CreditScore        10000 non-null  int64  
 4   Age                10000 non-null  int64  
 5   Tenure             10000 non-null  float64
 6   Balance            10000 non-

Debido a que tenemos 2 columnas con diferentes clasificaciones se hace one-hot encoding, para facilitar la clasificacion, ignorando la primera columna, en el caso de geography ya que si no es ningunga de las otras 2(0) se asume que es la tercera(1), al igual que en gender si no es male(0) se asume que es female(1).

In [5]:
#Entrenamos el modelo ignorando desbalance de clases
model = LogisticRegression(solver='liblinear', random_state=12345)
model.fit(features_train, target_train)
print('Modelo entrenado')

predict_valid = model.predict(features_valid)

#pruebas de calidad con valid
acc    = accuracy_score(target_valid, predict_valid)
recall = recall_score(target_valid, predict_valid)
f1     = f1_score(target_valid, predict_valid)

# --- probabilidades para ROC ---
prob_valid_lr = model.predict_proba(features_valid)[:, 1]

# curva y AUC
fpr_lr, tpr_lr, _ = roc_curve(target_valid, prob_valid_lr)
auc_lr = roc_auc_score(target_valid, prob_valid_lr)

# prints
print(f'El accuracy score es {acc}')
print(f'El recall score es {recall}')
print(f'El f1 score es {f1}')
print("ROC-AUC:", auc_lr)


Modelo entrenado
El accuracy score es 0.7828
El recall score es 0.06728971962616823
El f1 score es 0.11707317073170732
ROC-AUC: 0.6760847542270101




En el primer modelo entrenamos ignorarndo la desigaldad de clases que tiene una relacion de 80 a 20 o 4:1, como podemos ver el accuracy(porcentaje de aciertos totales) en este caso no es confiable ya que si predecimos que todos hacen churn(0), tenemos un accuracy de 80% pero sin que esto represente que realmente el modelo alla aprendido, si no simplemente porque el modelo repite la clase mayoritaria. 
En el caso de recall es de 0.05 lo cual es muy bajo, esto es el porcentaje de aciertos de lo ques realmetne hicieron churn(1), lo cual confirma que el recall no es confiable. 
    el caso de f1, es el balance entre ambas metricas anteriores, en este caso es de 0.09 lo cual nos dice que nuestro modelo tiene muy pocos aciertos combinados, lo cual se podria interpretar como que solo esta repitiendo la clase mayoritaria y no represante bien a la clase minoritaria(churn). 
        


In [6]:
#reentrenamos el modelo con balance para ver cambios en las métricas
model = LogisticRegression(solver='liblinear', class_weight='balanced', random_state=12345)
model.fit(features_train, target_train)
print('Modelo entrenado')

predict_valid = model.predict(features_valid)


Modelo entrenado


In [7]:
#pruebas de calidad: accuracy, recall, f1, roc auc
acc    = accuracy_score(target_valid, predict_valid)
recall = recall_score(target_valid, predict_valid)
f1     = f1_score(target_valid, predict_valid)

# --- probabilidades para ROC ---
prob_valid_lrb = model.predict_proba(features_valid)[:, 1]

# curva y AUC
fpr_lrb, tpr_lrb, _ = roc_curve(target_valid, prob_valid_lrb)
auc_lrb = roc_auc_score(target_valid, prob_valid_lrb)

# prints
print(f'El accuracy score es {acc}')
print(f'El recall score es {recall}')
print(f'El f1 score es {f1}')
print(f'El roc auc es {auc_lrb}')

El accuracy score es 0.6852
El recall score es 0.7046728971962617
El f1 score es 0.48929266709928615
El roc auc es 0.7565736843356876


En el segundo modelo podemos observar como el balance de clases vemos como afecta a la precision pero aumenta el recall, lo cual nos dice que representa de mejor manera a la clase minoritaria por lo cual el modelo realmente demuestra aprendizaje y no solo sesgo por la clae mayoritaria, de igual manera el f1 sube ya que si bien bajo el accuracy, la mejora en recall fue bastante considerable 

In [8]:
#entrenamos un modelo de RandomForestClassifier

#Buscamos el numero mas adecuado de estimadores y profundidad
best_f1 = 0
best_depth = None
best_estimators = None

for depth in range(1,16,1):
    for est in range(10,500,50):
        model = RandomForestClassifier(n_estimators=est, max_depth=depth, random_state=12345)
        model.fit(features_train, target_train)
        predict_valid= model.predict(features_valid)
        f1= f1_score(target_valid, predict_valid)
        
        if f1 > best_f1:
            best_f1= f1
            best_depth=depth
            best_estimators=est
        
    
    

print(best_f1)
print(best_depth)
print(best_estimators)



0.5795724465558194
15
360


In [9]:
#Entrenamos el modelo
model = RandomForestClassifier(n_estimators=300, max_depth=15, random_state=12345)
model.fit(features_train, target_train)
print('Modelo Entrenado!')

forest_valid = model.predict(features_valid)

#pruebas de calidad: f1, accuracy, recall, roc auc
forest_recall = recall_score(target_valid, forest_valid)
forest_acc    = accuracy_score(target_valid, forest_valid)
forest_f1     = f1_score(target_valid, forest_valid)

# --- probabilidades para ROC ---
prob_valid_rf = model.predict_proba(features_valid)[:, 1]

# curva y AUC
fpr_rf, tpr_rf, _ = roc_curve(target_valid, prob_valid_rf)
auc_rf = roc_auc_score(target_valid, prob_valid_rf)

# prints
print(f'El accuracy score es {forest_acc}')
print(f'El recall score es {forest_recall}')
print(f'El f1 score es {forest_f1}')
print(f'El roc auc es {auc_rf}')


Modelo Entrenado!
El accuracy score es 0.8576
El recall score es 0.44672897196261685
El f1 score es 0.5731414868105515
El roc auc es 0.858522270576205


In [10]:
En la fase de entrenamieto el bosque de clasificacion aleatoria obtiene mejores resultados tanto en accuracy como en recall, lo que significa que esta representando bien a ambas clases,  por tanto el f1 tambien es alto. 

SyntaxError: invalid syntax (1236782493.py, line 1)

In [ ]:
#grafica 
plt.figure(figsize=(6,6))
plt.plot(fpr_lr,  tpr_lr,  label=f'LogReg (AUC={auc_lr:.3f})')
plt.plot(fpr_lrb, tpr_lrb, label=f'LogReg balanced (AUC={auc_lrb:.3f})')
plt.plot(fpr_rf,  tpr_rf,  label=f'Random Forest (AUC={auc_rf:.3f})')
plt.plot([0,1],[0,1],'--', label='Azar (AUC=0.5)')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Curvas ROC (validación)')
plt.legend()
plt.grid(True)
plt.show()

Disminuye el sobreajuste que teniamos en accuracy, pero aun asi mantiene un buen desempeño, al igual que en recall, seguimos teninedo un buen desempeño por lo tanto se determina que es mucho mas eficiente para este caso un bosque de clasificacion aleatoria, sin embargo debemos tener en cuenta que dependiendo del tamaño del df, podria no ser la mejor opcion por el poder de computo que requiere. 
demás de accuracy, recall y F1, evaluamos ROC-AUC, que mide la capacidad global del modelo para separar clientes que se van de los que se quedan usando probabilidades, sin depender de un umbral fijo.
En este caso, el modelo de regresión logística sin balance apenas superaba el azar, mientras que la regresión logística balanceada alcanzó un ROC-AUC de ~0.73. El Random Forest mejoró ligeramente este valor hasta ~0.74.
Esto confirma que ambos modelos balanceados no solo obtuvieron mejores métricas de recall y F1, sino que también demostraron una mejor capacidad discriminatoria global respecto al modelo inicial.

<div class="alert alert-block alert-danger">
<b>Comentario del revisor</b> <a class="tocSkip"></a>

Entrenaste correctamente los modelos teniendo en cuenta el desequilibrio de clases. Ademeás usas diferentes sets de hiperparámetros para elegir los que mejor se ajustaran a los datos.

Solamente te recomiendo complementar con la gráfica ROC para evaluar el rendimiento del modelo. Este tipo de visualización permite comparar de forma efectiva distintos clasificadores y elegir el más adecuado.
</div>

<div class="alert alert-block alert-success">
<b>Comentario del revisor</b> <a class="tocSkip"></a>

Gran trabajo con el ajuste!